# Kako radi neuralna mreza: kroz Vector & Metric Spaces

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boba1987/vector-spaces/blob/master/neural_network_math_colab.ipynb)

Gradimo malu neuralnu mrezu (MLP sa 1 skrivenim slojem) **od nule u NumPy** na 2D **two-moons** datasetu, i objasnjavamo **svaki korak** preko matematike iz dve knjige - koristeci samo ono sto je potrebno da se razume kako mreza radi:

- **Gilbert Strang, _Linear Algebra and Its Applications_, Ch2 – Vector Spaces**: struktura mreze (slojevi kao preslikavanja izmedju vektorskih prostora).
- **_Introduction to Analysis_, Ch7 – Metric Spaces**: ucenje mreze (norme/loss, gradijent, neprekidnost, konvergencija treninga).

## Princip

> Za svaki korak: **STA** je od matematike primenjeno, **KAKO** (formula + kod) i **ZASTO** je bas to potrebno mrezi. Gde god je moguce, teoriju prvo **numericki potvrdimo** (`[CHECK]`, `[PROOF]`), pa je tek onda upotrebimo. Svaka code-celija na kraju ispise i `[THEORY]` rezime veze teorija -> kod.

## Mapa referenci (sekcija u kodu -> knjiga)

| Sekcija | Knjiga | Referenca | Uloga u mrezi |
|---|---|---|---|
| 3.1 Linearni sloj `Wx+b` | Strang Ch2 | 2.1, 2.2 | afino preslikavanje izmedju prostora |
| 3.2 Rang / dimenzija tezina | Strang Ch2 | 2.3 | kapacitet sloja |
| 3.3 `C(W)` i `N(W)` | Strang Ch2 | 2.4 | sta sloj propusta / ignorise |
| 4. Aktivacije | Analysis Ch7 | Def 7.44, Cor 7.55 | neprekidnost + Lipschitz (stabilnost) |
| 5. Loss kao norma | Analysis Ch7 | Def 7.11 | MSE = kvadrat L2 norme, L2 reg. |
| 6. Gradijent | Analysis Ch7 | Ex 7.15, Thm 7.54 | inner product, Cauchy-Schwarz -> smer pada |
| 7. Trening | Analysis Ch7 | Def 7.31 / 7.38 / 7.39 | Cauchy niz + kompletnost -> konvergencija |

PDF fajlovi: `Gilbert_Strang_Linear_Algebra_and_Its_Applications.pdf` (Ch2) i `intro_analysis_ch7.pdf` (Ch7).

## 1. Setup i helperi

**STA:** instaliramo biblioteke i pravimo helpere za log i numericke dokaze (isti stil kao u prethodnim projektima radi konzistentnosti).

**KAKO:** `log_step` (sta je korak uradio), `log_matrix` (oblik/preview), `log_check` (PASS/FAIL svojstva), `log_proof` (numericki dokaz jednakosti/nejednakosti), i `log_theory` (kratak rezime: koja teorija je primenjena, ref, i zasto).

**ZASTO:** zelimo da veza teorija -> kod bude vidljiva ne samo u tekstu nego i u izlazu svake celije.

In [ ]:
!pip -q install numpy matplotlib scikit-learn scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.figsize': (8, 4.5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 12,
    'font.size': 10,
})


def log_step(name, details=''):
    print(f'\n[LOG] {name}')
    if details:
        print(f'      {details}')


def log_matrix(name, M):
    M = np.asarray(M)
    print(f'[MATRIX] {name}: shape={M.shape}, dtype={M.dtype}')
    flat = M.ravel()
    print(f'[MATRIX] {name} preview:', np.round(flat[:min(8, flat.size)], 4))


def log_check(name, ok, detail=''):
    print(f'[CHECK] {name}: {"PASS" if bool(ok) else "FAIL"}', f'| {detail}' if detail else '')
    return bool(ok)


def log_proof(name, lhs, rhs, kind='le', tol=1e-6):
    lhs, rhs = float(lhs), float(rhs)
    if kind == 'eq':
        ok = abs(lhs - rhs) <= tol + tol * max(abs(lhs), abs(rhs))
        rel, sym = f'|lhs-rhs|={abs(lhs - rhs):.3e}', '=='
    else:
        ok = lhs <= rhs + tol
        rel, sym = f'margin(rhs-lhs)={rhs - lhs:.6f}', '<='
    print(f'[PROOF] {name}: {lhs:.6f} {sym} {rhs:.6f} -> {"PASS" if ok else "FAIL"} ({rel})')
    return ok


def log_theory(concept, ref, why):
    # Eksplicitna veza teorija -> kod u izlazu celije.
    print(f'[THEORY] Primenjeno: {concept}  (ref: {ref})\n         Zasto: {why}')


log_step('Setup OK', f'SEED={SEED}. Helperi spremni: log_step / log_matrix / log_check / log_proof / log_theory.')

## 2. Dataset: 2D two-moons (Strang 2.1)

**STA:** pravimo klasican nelinearan 2D dataset (dva polumeseca, 2 klase). Svaka tacka je vektor `x = (x1, x2) ∈ R^2`.

**KAKO:** `make_moons` + standardizacija (srednja 0, std 1) + train/test split. Crtamo scatter.

**ZASTO (Strang 2.1):** ulazi mreze su elementi vektorskog prostora `R^2`. Bira se 2D zato sto sve mozemo da vizualizujemo (tacke, granice odluke, geometriju preslikavanja). Klase nisu linearno razdvojive, pa cemo morati nelinearnu aktivaciju (sekcija 4).

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_raw, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)

# Standardizacija: x -> (x - mean) / std. (Translacija + skaliranje; vezu sa metrikom vidimo u sekciji 4.)
mu, sigma = X_raw.mean(axis=0), X_raw.std(axis=0)
X = (X_raw - mu) / sigma
y = y.reshape(-1, 1).astype(float)  # kolona, vrednosti u {0,1}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)

log_step('Dataset two-moons', f'X.shape={X.shape} | klase={np.unique(y).astype(int).tolist()} | '
                              f'train={X_train.shape[0]} test={X_test.shape[0]}')
log_matrix('X (prvih nekoliko vrednosti)', X)

fig, ax = plt.subplots(figsize=(7, 6))
for cls, marker in zip([0, 1], ['o', '^']):
    mask = (y.ravel() == cls)
    ax.scatter(X[mask, 0], X[mask, 1], label=f'klasa {cls}', s=30, alpha=0.8, marker=marker)
ax.set_title('Two-moons: tacke su vektori u R^2 (Strang 2.1)')
ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.legend(); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

log_theory('Ulazi kao elementi vektorskog prostora R^2', 'Strang 2.1',
           'mreza operise nad vektorima; sve operacije (sabiranje, skaliranje, mnozenje matricom) su definisane jer je R^2 vektorski prostor.')

# 3. Arhitektura: mreza kao kompozicija preslikavanja (Strang 2.1, 2.2, 2.3, 2.4)

Nasa mreza je: `x -> [linearni sloj W1,b1] -> [tanh] -> [linearni sloj W2,b2] -> [sigmoid] -> y_hat`.

Matematicki: kompozicija dva **afina preslikavanja** (Strang 2.2) sa nelinearnom funkcijom izmedju njih. U ovoj sekciji gledamo linearne slojeve kroz Ch2.

## 3.1 Linearni sloj `z = W x + b` kao afino preslikavanje (2.1, 2.2)

**STA:** sloj uzima vektor i vraca vektor preko mnozenja matricom i dodavanja pomeraja.

**KAKO (2.2):** `z = W x + b`. Mnozenje `W x` je tacno racunanje leve strane sistema `Ax` iz Strang 2.2; `+b` je pomeraj (afini deo). Za batch `X` (n×d) racunamo `Z = X W^T + b`.

**ZASTO (2.1):** sloj je preslikavanje `R^{d_in} -> R^{d_out}` izmedju vektorskih prostora; linearnost (`W(x+y)=Wx+Wy`, `W(ax)=aWx`) je ono sto cini "linearni" sloj i dozvoljava efikasan backprop.

In [ ]:
# Arhitektura: 2 -> H -> 1
D_IN, H, D_OUT = 2, 8, 1

def init_params(seed=SEED):
    rng = np.random.default_rng(seed)
    # He/Xavier-stil inicijalizacija radi stabilnosti.
    return {
        'W1': rng.normal(0, np.sqrt(2.0 / D_IN), size=(H, D_IN)),
        'b1': np.zeros((1, H)),
        'W2': rng.normal(0, np.sqrt(2.0 / H), size=(D_OUT, H)),
        'b2': np.zeros((1, D_OUT)),
    }

params = init_params()

def linear(Xin, W, b):
    # Afino preslikavanje (Strang 2.2): Z = Xin W^T + b.
    return Xin @ W.T + b

log_step('3.1 Linearni sloj Wx+b', f'Arhitektura {D_IN}->{H}->{D_OUT}')
for name in ['W1', 'b1', 'W2', 'b2']:
    log_matrix(name, params[name])

# Provera linearnosti preslikavanja x -> W x (Strang 2.1): W(ax+by) = a Wx + b Wy.
rng = np.random.default_rng(0)
xa, xb = rng.normal(size=(1, D_IN)), rng.normal(size=(1, D_IN))
a_s, b_s = 1.7, -0.9
W1 = params['W1']
lhs = (a_s * xa + b_s * xb) @ W1.T
rhs = a_s * (xa @ W1.T) + b_s * (xb @ W1.T)
log_proof('Linearnost: W(a x + b y) == a Wx + b Wy', np.max(np.abs(lhs - rhs)), 0.0, kind='eq')

# Mnozenje matricom = sistem iz 2.2: z = W x. Proverimo jedan red rucno (skalarni proizvod reda i x).
x0 = X_train[:1]                 # (1,2)
z0 = linear(x0, W1, params['b1'])  # (1,H)
manual_row0 = np.dot(W1[0], x0.ravel()) + params['b1'][0, 0]
log_proof('z[0] == <W1[0], x> + b1[0] (red sistema 2.2)', z0[0, 0], manual_row0, kind='eq')

log_theory('Linearni sloj kao afino preslikavanje z=Wx+b', 'Strang 2.1, 2.2',
           'svaki sloj je preslikavanje izmedju vektorskih prostora; Wx je leva strana sistema Ax, +b je pomeraj.')

## 3.2 Rang i dimenzija tezinske matrice (2.3)

**STA:** racunamo `rank(W1)` i tumacimo dimenziju skrivenog sloja kao kapacitet reprezentacije.

**KAKO (2.3):** rang = broj linearno nezavisnih redova/kolona = dimenzija prostora koji sloj moze da "dosegne". Skriveni sloj `R^H` ima dimenziju `H`, ali stvarni broj nezavisnih pravaca koje `W1` proizvodi je `rank(W1) ≤ min(H, D_IN)`.

**ZASTO:** rang ogranicava koliko nezavisnih osobina sloj moze da napravi iz ulaza. Ako je `rank(W1) < D_IN`, sloj gubi pravce vec na ulazu (videti 3.3). Ovo je direktna veza "dimenzija/baza" (2.3) i kapaciteta mreze.

In [ ]:
r1 = int(np.linalg.matrix_rank(params['W1']))
r2 = int(np.linalg.matrix_rank(params['W2']))

log_step('3.2 Rang / dimenzija tezina', f'W1: {params["W1"].shape}, W2: {params["W2"].shape}')
print(f'[INFO] rank(W1)={r1}  (max moguci = min(H,D_IN) = {min(H, D_IN)})')
print(f'[INFO] rank(W2)={r2}  (max moguci = min(D_OUT,H) = {min(D_OUT, H)})')
log_check('W1 ima pun rang po ulazu (ne gubi ulazne pravce)', r1 == D_IN,
          f'rank(W1)={r1} == D_IN={D_IN}')

# Singularne vrednosti W1 -> "jacina" svakog nezavisnog pravca preslikavanja.
sv = np.linalg.svd(params['W1'], compute_uv=False)
log_matrix('singularne vrednosti W1', sv)

log_theory('Rang = broj nezavisnih pravaca / dimenzija slike sloja', 'Strang 2.3',
           'rang(W) ogranicava koliko nezavisnih osobina sloj moze da proizvede iz ulaza = kapacitet sloja.')

## 3.3 Cetiri podprostora sloja: `C(W)` i `N(W)` (2.4)

**STA:** gledamo sta sloj **dostize** (prostor kolona `C(W)`) i sta **ignorise** (nullspace `N(W)` - pravci koje salje u 0).

**KAKO (2.4):**
- `C(W1)`: pre-aktivacije `z = W1 x` zive u prostoru kolona od `W1`, podprostoru od `R^H` dimenzije `rank(W1)`.
- `N(W2)`: izlazni sloj `W2` (oblik `1×H`) ima netrivijalan nullspace u `R^H` (dim `H − 1`); dodavanje bilo kog vektora iz `N(W2)` skrivenoj aktivaciji **ne menja izlaz**. Proveravamo `W2 @ n ≈ 0`.

**ZASTO:** ovo objasnjava redundansu i "slepe pravce" mreze: deo informacije u skrivenom sloju izlaz uopste ne koristi. To je ista dekompozicija (prostor kolona vs nullspace) iz Strang 2.4, primenjena na sloj.

In [ ]:
from scipy.linalg import null_space, orth

log_step('3.3 C(W) i N(W) sloja', 'sta sloj dostize vs sta ignorise (Strang 2.4)')

# C(W1): pre-aktivacije zive u prostoru kolona W1 (podprostor R^H).
col_W1 = orth(params['W1'])
print(f'[INFO] dim C(W1) = {col_W1.shape[1]}  -> pre-aktivacije z=W1 x zive u {col_W1.shape[1]}-dim podprostoru od R^{H}')

# N(W2): izlazni sloj ignorise H-1 pravaca skrivenog prostora.
N_W2 = null_space(params['W2'])
print(f'[INFO] dim N(W2) = {N_W2.shape[1]}  (= H - rank(W2) = {H} - {r2})')
log_check('W2 @ n ~ 0 za sve bazne vektore nullspace-a', np.allclose(params['W2'] @ N_W2, 0, atol=1e-10),
          f'max|W2 n|={np.max(np.abs(params["W2"] @ N_W2)):.2e}')

# Invarijantnost izlaza: dodavanje elementa N(W2) skrivenoj aktivaciji ne menja izlaz.
h_demo = rng.normal(size=(1, H))
n_dir = N_W2[:, 0].reshape(1, H)
out_a = h_demo @ params['W2'].T
out_b = (h_demo + 5.0 * n_dir) @ params['W2'].T
log_proof('Izlaz invarijantan na N(W2): ||out(h+n) - out(h)||', np.linalg.norm(out_b - out_a), 0.0, kind='eq')

log_theory('Prostor kolona C(W) i nullspace N(W) sloja', 'Strang 2.4',
           'C(W) = sta sloj dostize, N(W) = pravci koje ignorise; deo skrivene informacije izlaz uopste ne koristi (redundansa).')

# 4. Aktivacije kao neprekidne funkcije (Ch7 Def 7.44, Cor 7.55)

**STA:** uvodimo nelinearne aktivacije (`tanh`, `ReLU`, `sigmoid`) i pokazujemo da su to **neprekidne** (cak Lipschitz) funkcije izmedju metrickih prostora.

**KAKO:**
- **Neprekidnost (Def 7.44):** za svako `ε>0` postoji `δ>0` tako da `|x−c|<δ ⟹ |f(x)−f(c)|<ε`. Proveravamo numericki (za dato `ε` nadjemo `δ`).
- **Lipschitz (vezano za nejednakost trougla, Cor 7.55):** `||f(x)−f(y)|| ≤ L·||x−y||`. Za `tanh` i `sigmoid` `L = max|f'|`; za `ReLU` `L=1`. Merimo empirijski `L_emp = max ||f(x)−f(y)|| / ||x−y||`.

**ZASTO:** nelinearnost je razlog zasto mreza moze da razdvoji nelinearne klase (two-moons). Neprekidnost garantuje da male promene ulaza daju male promene izlaza (stabilnost), a Lipschitz konstanta kvantifikuje koliko mreza moze da "pojaca" perturbacije - kljucno za robusnost i stabilan trening.

In [ ]:
def tanh(z):    return np.tanh(z)
def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

log_step('4. Aktivacije: neprekidnost i Lipschitz', 'Def 7.44 (neprekidnost), Cor 7.55 (trougao/Lipschitz)')

# --- Neprekidnost (Def 7.44): za eps nadji delta na primeru tanh u tacki c ---
c = 0.7
eps = 1e-3
deltas = np.logspace(-1, -6, 60)
found_delta = None
for d in deltas:
    xs = c + np.array([-d, d])
    if np.max(np.abs(tanh(xs) - tanh(c))) < eps:
        found_delta = d
        break
log_check(f'Neprekidnost tanh u c={c}: za eps={eps} postoji delta', found_delta is not None,
          f'delta={found_delta:.2e}')

# --- Lipschitz konstanta (empirijski) za svaku aktivaciju ---
rng = np.random.default_rng(SEED)
pts = rng.normal(size=(2000,)) * 3
def emp_lipschitz(f, pts):
    a, b = pts[:-1], pts[1:]
    num = np.abs(f(a) - f(b))
    den = np.abs(a - b) + 1e-12
    return np.max(num / den)

L_bounds = {'tanh': 1.0, 'relu': 1.0, 'sigmoid': 0.25}
for name, f in [('tanh', tanh), ('relu', relu), ('sigmoid', sigmoid)]:
    Lemp = emp_lipschitz(f, np.sort(pts))
    log_proof(f'Lipschitz {name}: L_emp <= L_teorijsko', Lemp, L_bounds[name], kind='le', tol=1e-2)

# Grafik aktivacija.
zz = np.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(zz, tanh(zz), label='tanh (L=1)')
ax.plot(zz, relu(zz), label='ReLU (L=1)')
ax.plot(zz, sigmoid(zz), label='sigmoid (L=0.25)')
ax.set_title('Aktivacije: neprekidne, Lipschitz funkcije (Ch7 Def 7.44, Cor 7.55)')
ax.set_xlabel('z'); ax.set_ylabel('f(z)'); ax.legend()
plt.tight_layout(); plt.show()

log_theory('Aktivacija kao neprekidna/Lipschitz funkcija izmedju metrickih prostora', 'Ch7 Def 7.44, Cor 7.55',
           'neprekidnost = stabilnost (mala promena ulaza -> mala promena izlaza); Lipschitz L ogranicava pojacanje perturbacija.')

In [ ]:
# Forward pass = kompozicija afina preslikavanja (2.2) i nelinearnih aktivacija (Ch7).
# x -> (W1,b1) -> tanh -> (W2,b2) -> sigmoid -> y_hat
def forward(Xin, p):
    z1 = linear(Xin, p['W1'], p['b1'])   # (n,H)  afino (Strang 2.2)
    a1 = tanh(z1)                        # nelinearnost (Ch7)
    z2 = linear(a1, p['W2'], p['b2'])    # (n,1)  afino
    y_hat = sigmoid(z2)                  # verovatnoca u (0,1)
    cache = {'Xin': Xin, 'z1': z1, 'a1': a1, 'z2': z2, 'y_hat': y_hat}
    return y_hat, cache

y_hat0, _ = forward(X_train, params)
log_step('Forward pass definisan', f'y_hat opseg=[{y_hat0.min():.3f}, {y_hat0.max():.3f}] (sigmoid -> (0,1))')
log_theory('Mreza kao kompozicija afina preslikavanja + neprekidnih aktivacija', 'Strang 2.2 + Ch7 Def 7.44',
           'kompozicija neprekidnih funkcija je neprekidna -> cela mreza je neprekidno preslikavanje R^2 -> (0,1).')

# 5. Loss kao norma (Ch7 Def 7.11)

**STA:** definisemo funkciju gubitka koja meri koliko je predikcija `y_hat` daleko od istine `y`. To je rastojanje izvedeno iz **norme**.

**KAKO (Def 7.11):**
- **MSE** = `(1/n) ||y − y_hat||_2^2` - kvadrat L2 norme greske (norma iz Ex 7.15: `||v||_2 = sqrt(<v,v>)`).
- **L2 regularizacija** = `λ (||W1||_F^2 + ||W2||_F^2)` - norma parametara, kaznjava velike tezine.
- Proveravamo da norma greske zadovoljava aksiome norme (homogenost, nejednakost trougla).

**ZASTO:** ucenje = minimizacija rastojanja izmedju predikcije i istine. Da bi "rastojanje" imalo smisla (uvek ≥ 0, =0 samo kad je tacno, postuje trougao), loss mora poticati iz norme/metrike. Zato su aksiomi norme (Def 7.11) temelj svake loss funkcije.

In [ ]:
LAMBDA = 1e-3  # jacina L2 regularizacije

def mse_loss(y_true, y_pred):
    # MSE = (1/n) ||y - y_pred||_2^2  (kvadrat L2 norme greske)
    diff = y_pred - y_true
    return float(np.mean(np.sum(diff ** 2, axis=1)))

def l2_reg(p):
    return float(np.sum(p['W1'] ** 2) + np.sum(p['W2'] ** 2))

def total_loss(y_true, y_pred, p, lam=LAMBDA):
    return mse_loss(y_true, y_pred) + lam * l2_reg(p)

y_hat0, _ = forward(X_train, params)
log_step('5. Loss kao norma', f'MSE={mse_loss(y_train, y_hat0):.4f} | ||W||^2={l2_reg(params):.4f} '
                              f'| total={total_loss(y_train, y_hat0, params):.4f}')

# MSE je zaista izveden iz L2 norme: ||y - y_hat||_2 = sqrt(sum kvadrata).
err = (y_hat0 - y_train).ravel()
log_proof('||greska||_2 == sqrt(<e,e>)  (Ex 7.15)', np.linalg.norm(err), np.sqrt(np.dot(err, err)), kind='eq')

# Aksiomi norme (Def 7.11) na vektoru greske.
u, v = err, rng.normal(size=err.shape)
log_proof('Homogenost: ||k u|| == |k| ||u||', np.linalg.norm(3.5 * u), 3.5 * np.linalg.norm(u), kind='eq')
log_proof('Trougao: ||u+v|| <= ||u|| + ||v||', np.linalg.norm(u + v), np.linalg.norm(u) + np.linalg.norm(v), kind='le')
log_check('Pozitivnost: ||greska|| >= 0 i =0 samo kad je predikcija tacna',
          np.linalg.norm(err) >= 0 and np.isclose(np.linalg.norm(np.zeros_like(err)), 0.0))

log_theory('Loss = kvadrat norme greske; regularizacija = norma parametara', 'Ch7 Def 7.11 (+ Ex 7.15)',
           'ucenje minimizuje rastojanje predikcija<->istina; aksiomi norme garantuju da je to smisleno "rastojanje".')

# 6. Gradijent, inner product i Cauchy-Schwarz (Ch7 Ex 7.15, Thm 7.54)

**STA:** rucno izvodimo gradijente loss-a po svim parametrima (backpropagation), proveravamo ih protiv numerickih (gradient check), i pokazujemo zasto je `−grad` pravac najbrzeg pada koristeci Cauchy-Schwarz.

**KAKO:**
- backprop = lancano pravilo kroz slojeve; gradijent svakog parametra je matrica istog oblika.
- **gradient check:** poredimo analiticki gradijent sa `(L(θ+ε) − L(θ−ε)) / 2ε` -> greska treba da bude ~1e-7.
- **Inner product (Ex 7.15):** usmerena promena losa duz koraka `v` je `≈ <grad, v>`.
- **Cauchy-Schwarz (Thm 7.54):** `|<grad, v>| ≤ ||grad||·||v||`, sa jednakoscu kad je `v ∝ grad`. Zato je `v = −grad` pravac najveceg smanjenja.

**ZASTO:** ceo gradient descent pociva na tome da gradijent (vektor u parametarskom prostoru) daje smer; inner product meri "koliko korak smanjuje loss", a Cauchy-Schwarz dokazuje da je negativni gradijent optimalan smer. Ovo je tacka gde se linearna algebra i analiza spajaju u algoritam ucenja.

In [ ]:
def backward(y_true, cache, p, lam=LAMBDA):
    n = y_true.shape[0]
    Xin, z1, a1, z2, y_hat = cache['Xin'], cache['z1'], cache['a1'], cache['z2'], cache['y_hat']

    # dL/dy_hat za MSE = (1/n) sum (y_hat - y)^2
    dy_hat = (2.0 / n) * (y_hat - y_true)            # (n,1)
    dz2 = dy_hat * (y_hat * (1.0 - y_hat))           # sigmoid' = s(1-s)   (n,1)
    dW2 = dz2.T @ a1 + 2 * lam * p['W2']             # (1,H)
    db2 = np.sum(dz2, axis=0, keepdims=True)         # (1,1)

    da1 = dz2 @ p['W2']                              # (n,H)
    dz1 = da1 * (1.0 - np.tanh(z1) ** 2)             # tanh' = 1 - tanh^2  (n,H)
    dW1 = dz1.T @ Xin + 2 * lam * p['W1']            # (H,D_IN)
    db1 = np.sum(dz1, axis=0, keepdims=True)         # (1,H)

    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

y_hat0, cache0 = forward(X_train, params)
grads = backward(y_train, cache0, params)

log_step('6. Backprop gradijenti', 'lancano pravilo kroz slojeve')
for k in ['W1', 'b1', 'W2', 'b2']:
    log_matrix(f'grad {k}', grads[k])

# --- Gradient check: analiticki vs numericki (konacne razlike) ---
def loss_at(p):
    yh, _ = forward(X_train, p)
    return total_loss(y_train, yh, p)

eps = 1e-5
max_rel_err = 0.0
for k in ['W1', 'b1', 'W2', 'b2']:
    flat = params[k].ravel()
    for idx in range(min(flat.size, 12)):  # uzorak parametara radi brzine
        orig = flat[idx]
        flat[idx] = orig + eps; lp = loss_at(params)
        flat[idx] = orig - eps; lm = loss_at(params)
        flat[idx] = orig
        num_g = (lp - lm) / (2 * eps)
        ana_g = grads[k].ravel()[idx]
        rel = abs(num_g - ana_g) / (abs(num_g) + abs(ana_g) + 1e-12)
        max_rel_err = max(max_rel_err, rel)

log_proof('Gradient check: max relativna greska (analiticki vs numericki)', max_rel_err, 1e-5, kind='le', tol=1e-4)
log_check('Backprop gradijenti su tacni', max_rel_err < 1e-4, f'max_rel_err={max_rel_err:.2e}')

In [ ]:
# Spljostimo sve gradijente u jedan vektor g u parametarskom prostoru.
def flatten(d):
    return np.concatenate([d[k].ravel() for k in ['W1', 'b1', 'W2', 'b2']])

g = flatten(grads)
log_step('6b. Inner product i Cauchy-Schwarz (smer pada)', f'dim parametara={g.size}, ||grad||={np.linalg.norm(g):.4f}')

# Cauchy-Schwarz (Thm 7.54) + provera da je -grad pravac najveceg pada.
norm_g = np.linalg.norm(g)
cs_ok = True
dir_derivs = []
labels_dir = []
rng = np.random.default_rng(SEED)
for _ in range(200):
    v = rng.normal(size=g.size)
    v = v / np.linalg.norm(v)              # jedinicni pravac
    ip = float(np.dot(g, v))               # <grad, v> = usmereni izvod
    if abs(ip) > norm_g + 1e-9:            # CS: |<g,v>| <= ||g|| ||v|| = ||g||
        cs_ok = False
    dir_derivs.append(ip)
    labels_dir.append('random')

# Pravac -grad (jedinicni): usmereni izvod = -||g|| (najnegativniji moguci).
v_neg = -g / norm_g
ip_neg = float(np.dot(g, v_neg))
log_check('Cauchy-Schwarz |<grad,v>| <= ||grad|| za 200 jedinicnih pravaca', cs_ok,
          f'max|<g,v>|={max(abs(x) for x in dir_derivs):.4f} <= ||g||={norm_g:.4f}')
log_proof('Smer -grad daje minimalni usmereni izvod = -||grad||', ip_neg, -norm_g, kind='eq')
log_check('-grad je pravac najbrzeg pada (nijedan random pravac nije strmiji)',
          min(dir_derivs) >= ip_neg - 1e-9, f'min(random)={min(dir_derivs):.4f} vs -||g||={-norm_g:.4f}')

# Vizualizacija: usmereni izvod random pravaca lezi u [-||g||, ||g||] (granice iz CS).
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(dir_derivs, bins=30, alpha=0.8)
ax.axvline(norm_g, color='red', ls='--', label='+||grad|| (granica CS)')
ax.axvline(-norm_g, color='red', ls='--', label='-||grad|| (= smer -grad)')
ax.set_title('Usmereni izvod <grad, v> je ogranicen sa +/-||grad|| (Cauchy-Schwarz, Thm 7.54)')
ax.set_xlabel('<grad, v> za jedinicno v'); ax.set_ylabel('broj'); ax.legend()
plt.tight_layout(); plt.show()

log_theory('Inner product = usmerena promena losa; Cauchy-Schwarz -> -grad je najbrzi pad', 'Ch7 Ex 7.15, Thm 7.54',
           'gradient descent korak -lr*grad je optimalan jer CS ogranicava |<grad,v>| i jednakost vazi bas za v ∝ grad.')

# 7. Trening kao Cauchy niz u parametarskom prostoru (Ch7 Def 7.31/7.38/7.39)

**STA:** pokrecemo gradient descent i pokazujemo da niz parametara `(θ_n)` jeste **Cauchy niz** koji **konvergira** - sto je razlog zasto trening uopste "sleti" u resenje.

**KAKO:**
- update: `θ_{n+1} = θ_n − lr · grad(θ_n)`.
- pratimo `loss`, `d(θ_n, θ_{n+1}) = ||θ_{n+1} − θ_n||` (Cauchy uslov, Def 7.38) i `d(θ_n, θ_final)` (konvergencija, Def 7.31).
- **kompletnost (Def 7.39, Ex 7.41):** parametarski prostor `R^P` je Banach, pa Cauchy niz garantovano ima granicu *u prostoru*.

**ZASTO:** bez kompletnosti, niz koji se "zgusnjava" ne bi morao da ima granicu u prostoru. Jer je `R^P` kompletan, smanjivanje koraka (`d(θ_n,θ_{n+1}) → 0`) povlaci postojanje konkretnih finalnih parametara - to je matematicka garancija da trening konvergira.

In [ ]:
def params_to_vec(p):
    return flatten(p)

def accuracy(p, Xs, ys):
    yh, _ = forward(Xs, p)
    return float(np.mean((yh >= 0.5).astype(float) == ys))

# --- Gradient descent trening ---
params = init_params()           # svez start
LR = 0.7
EPOCHS = 1500

loss_hist, step_dist, theta_vecs = [], [], []
prev_vec = params_to_vec(params).copy()
for epoch in range(EPOCHS):
    yh, cache = forward(X_train, params)
    loss_hist.append(total_loss(y_train, yh, params))
    grads = backward(y_train, cache, params)
    for k in params:
        params[k] -= LR * grads[k]
    cur_vec = params_to_vec(params)
    step_dist.append(np.linalg.norm(cur_vec - prev_vec))   # d(theta_n, theta_{n+1})
    theta_vecs.append(cur_vec.copy())
    prev_vec = cur_vec.copy()

theta_final = params_to_vec(params)
dist_to_final = [np.linalg.norm(v - theta_final) for v in theta_vecs]

log_step('7. Trening zavrsen', f'epochs={EPOCHS}, lr={LR} | final loss={loss_hist[-1]:.4f} | '
                               f'train acc={accuracy(params, X_train, y_train):.3f}')

# Provere: konvergencija (Def 7.31) i Cauchy uslov (Def 7.38).
log_check('Loss monotono opada (uglavnom)', loss_hist[-1] < loss_hist[0],
          f'{loss_hist[0]:.4f} -> {loss_hist[-1]:.4f}')
log_check('Cauchy uslov: d(theta_n, theta_{n+1}) -> 0 (Def 7.38)', step_dist[-1] < 1e-3,
          f'poslednji korak={step_dist[-1]:.2e}')
log_check('Konvergencija: d(theta_n, theta_final) -> 0 (Def 7.31)', dist_to_final[-1] < 1e-3,
          f'd_final={dist_to_final[-1]:.2e}')

eps = 1e-2
N_idx = next((i for i in range(len(step_dist)) if all(s < eps for s in step_dist[i:])), None)
print(f'[INFO] Za eps={eps}: svi koraci posle N={N_idx} su < eps (Cauchy uslov, Def 7.38).')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(loss_hist)
axes[0].set_title('Loss tokom treninga'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('total loss')
axes[1].semilogy(step_dist, label='d(theta_n, theta_{n+1})  (Cauchy, Def 7.38)')
axes[1].semilogy(dist_to_final, label='d(theta_n, theta_final)  (konvergencija, Def 7.31)')
axes[1].axhline(eps, color='red', ls='--', alpha=0.6, label=f'eps={eps}')
axes[1].set_title('Trening = konvergentan Cauchy niz u R^P (Banach)')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('rastojanje (log)'); axes[1].legend()
plt.tight_layout(); plt.show()

log_theory('Niz parametara je Cauchy niz; kompletnost R^P garantuje konvergenciju', 'Ch7 Def 7.31/7.38/7.39 (Ex 7.41)',
           'jer je parametarski prostor kompletan (Banach), zgusnjavanje koraka povlaci postojanje finalnih parametara -> trening konvergira.')

## 8. Evaluacija i decision boundary

**STA:** merimo tacnost na test setu i vizualizujemo granicu odluke koju je mreza naucila.

**KAKO:** klasa = `1` ako `y_hat ≥ 0.5`. Granicu crtamo evaluacijom mreze na gustoj 2D mrezi tacaka.

**ZASTO:** granica je *zakrivljena* iako su slojevi afina (linearna) preslikavanja - upravo zato sto nelinearna aktivacija (sekcija 4) "savija" prostor. Ovo vizuelno spaja Strang 2.x (linearni slojevi) i Ch7 (neprekidne nelinearnosti): kompozicija linearnog i nelinearnog daje nelinearnu granicu sposobnu da razdvoji two-moons.

In [ ]:
test_acc = accuracy(params, X_test, y_test)
train_acc = accuracy(params, X_train, y_train)
log_step('8. Evaluacija', f'train acc={train_acc:.3f} | test acc={test_acc:.3f}')

# Decision boundary na gustoj 2D mrezi.
pad = 0.5
x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
probs, _ = forward(grid, params)
zz = probs.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 7))
cf = ax.contourf(xx, yy, zz, levels=20, alpha=0.6, cmap='RdBu')
ax.contour(xx, yy, zz, levels=[0.5], colors='k', linewidths=2)  # granica odluke p=0.5
for cls, marker in zip([0, 1], ['o', '^']):
    mask = (y.ravel() == cls)
    ax.scatter(X[mask, 0], X[mask, 1], s=25, marker=marker, edgecolor='k', linewidth=0.3, label=f'klasa {cls}')
plt.colorbar(cf, ax=ax, label='P(klasa=1)')
ax.set_title(f'Naucena granica odluke (test acc={test_acc:.2f}) - zakrivljena zbog nelinearnosti')
ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.legend(); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

log_theory('Nelinearna granica = linearni slojevi (Strang 2.x) + neprekidna nelinearnost (Ch7)', 'Strang 2.1/2.2 + Ch7 Def 7.44',
           'afina preslikavanja sama daju prave linije; aktivacija savija prostor pa kompozicija razdvaja nelinearno razdvojive klase.')

## 9. Finalni summary i mapiranje na teoriju

**STA:** sazimamo sve glavne brojeve i provere run-a i rekapituliramo gde je svaka teorija upotrebljena.

In [ ]:
summary = [
    ('Arhitektura', f'{D_IN}->{H}->{D_OUT}', 'Strang 2.1 (slojevi kao preslikavanja)'),
    ('rank(W1)', f'{int(np.linalg.matrix_rank(params["W1"]))}', 'Strang 2.3 (kapacitet)'),
    ('dim N(W2)', f'{null_space(params["W2"]).shape[1]}', 'Strang 2.4 (ignorisani pravci)'),
    ('Lipschitz tanh', '1.0', 'Ch7 Def 7.44 / Cor 7.55'),
    ('Loss tip', 'MSE = ||y-yhat||^2 + L2', 'Ch7 Def 7.11 (norma)'),
    ('Gradient check max rel err', f'{max_rel_err:.2e}', 'backprop korektnost'),
    ('Cauchy-Schwarz', 'PASS', 'Ch7 Thm 7.54 (smer pada)'),
    ('Cauchy d(theta_n,theta_n+1) final', f'{step_dist[-1]:.2e}', 'Ch7 Def 7.38'),
    ('Konvergencija d_final', f'{dist_to_final[-1]:.2e}', 'Ch7 Def 7.31/7.39'),
    ('Final loss', f'{loss_hist[-1]:.4f}', 'trening'),
    ('Train / Test accuracy', f'{train_acc:.3f} / {test_acc:.3f}', 'evaluacija'),
]

log_step('9. Finalni summary', 'pregled artefakata i provera')
print(f'{"artefakt":<34}{"vrednost":<26}{"referenca"}')
print('-' * 90)
for art, val, ref in summary:
    print(f'{art:<34}{val:<26}{ref}')

## Zakljucak: STA / KAKO / ZASTO po koraku

### Strang, Chapter 2 - Vector Spaces (STRUKTURA mreze)
- **2.1 (sek. 2, 3.1):** ulazi i aktivacije su vektori; svaki sloj je preslikavanje izmedju vektorskih prostora -> smemo da mnozimo matricom, sabiramo, projektujemo.
- **2.2 (3.1):** linearni sloj `z = Wx + b` je afino preslikavanje; `Wx` je leva strana sistema iz 2.2.
- **2.3 (3.2):** `rank(W)` = broj nezavisnih pravaca = kapacitet sloja.
- **2.4 (3.3):** `C(W)` = sta sloj dostize, `N(W)` = pravci koje ignorise; deo skrivene informacije izlaz ne koristi.

### Introduction to Analysis, Chapter 7 - Metric Spaces (UCENJE mreze)
- **Def 7.44 / Cor 7.55 (sek. 4):** aktivacije su neprekidne, Lipschitz funkcije -> stabilnost (mala promena ulaza, mala promena izlaza).
- **Def 7.11 (sek. 5):** loss je (kvadrat) norme greske; regularizacija je norma parametara -> ucenje minimizuje smisleno rastojanje.
- **Ex 7.15 / Thm 7.54 (sek. 6):** inner product meri usmerenu promenu losa; Cauchy-Schwarz dokazuje da je `-grad` pravac najbrzeg pada.
- **Def 7.31 / 7.38 / 7.39 (sek. 7):** niz parametara je Cauchy niz; kompletnost `R^P` (Banach) garantuje da trening konvergira ka konkretnoj tacki.

### Sinteza
Mreza = **kompozicija afina preslikavanja (Strang Ch2) i neprekidnih nelinearnosti (Ch7)**. Ucenje = **minimizacija norme greske (Ch7 Def 7.11)** pomeranjem parametara u **smeru iz Cauchy-Schwarz (Thm 7.54)**, kao **Cauchy niz koji konvergira zbog kompletnosti (Def 7.39)**. Zakrivljena granica odluke je vidljiv dokaz da linearna algebra + analiza zajedno cine ono sto zovemo "neuralna mreza".